**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/van-der-molen-2025"], check=True)
    os.chdir("/content/encore/entries/van-der-molen-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Hidden Markov models: van der Molen et al. (2025) on dandiset 001603

The HMM claims of the paper (Figs. 5 and 7F, Supplementary Figs. 18, 24D and 26), on
[DANDI:001603](https://dandiarchive.org/dandiset/001603/0.260923.0623) version 0.260923.0623,
the only published version (see the header of `reproduce_van_der_molen_2025.ipynb` for the
choice of version and the data check).

**Model.** Poisson-observation HMMs on spike counts in 30 ms bins (Supplementary Methods), fitted
by EM in `hmm.py`, which implements the model of the SSM package the paper used (SSM does not
install from PyPI in a locked environment). The recording lengths are the code's (180 s or 600 s),
burst peaks and backbone units are as in the main notebook (`sequences.py`).

**Deviations for runtime.** The paper fits every K from 10 to 30 (21 models per recording); this
notebook fits K = 10, 12, ..., 30 (11 models per recording), with at most 300 EM iterations where
the code allows 5000. The cross-validation of Supplementary Fig. 18 uses K = 10, 20 and 30.

| Claim | Tested here |
|---|---|
| supp18-hmm-likelihood-real-above-shuffled | tested |
| fig5c-states-converge-at-peak | tested |
| fig5d-backbone-more-states | tested |
| fig5f-svm-states-beat-rate | tested |
| fig7f-hmm-dimensionality | tested |
| supp24d-backbone-non-poisson | tested |
| supp26-state-traversal | tested |

In [ ]:
import os

for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[v] = "1"  # one linear-algebra thread per process: the fits run in parallel processes
import time
import warnings

import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
from scipy import stats

import hmm
import sequences as sq

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
started = time.time()
assets = {a["path"]: a for a in sq.list_assets()}
KS = list(range(10, 31, 2))
CV_KS = [10, 20, 30]

In [ ]:
sorted_paths = sorted(p for p in assets if "_ecephys" not in p)
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(8) as ex:
    infos = list(ex.map(sq.sorted_file_info, [assets[p] for p in sorted_paths]))
files = sq.recording_table(list(assets.values()), infos)
first = files.sort_values("path").groupby("recording").first()


def fit_recording(name):
    """Bursts, backbone units and HMMs (every K) of one recording."""
    M = sq.load_matrix(first.loc[name].asset_id, sq.keep_seconds(name))
    mt = sq.model_type(name)
    b = sq.detect_bursts(M, mt)
    out = dict(name=name, mtype=mt, U=M.shape[0], T=M.shape[1], n_bursts=len(b["peak"]),
               rate=M.sum(1) / (M.shape[1] / 1000))
    if len(b["peak"]) >= 2:
        counts = sq.spikes_in_bursts(M, b)
        out["bb"] = (counts >= 2).mean(1) >= sq.BACKBONE_FRAC[mt] - 1e-12
        R = sq.isi_rates(M)
        P = sq.peak_times(R, b, counts)
        del R
        out["window"] = sq.backbone_window(P, out["bb"]) if out["bb"].sum() else None
        out["bursts"] = {k: b[k] for k in ("start", "end", "peak")}
    X = hmm.bin_counts(M)
    out["X"] = X
    out["models"] = {}
    for K in KS:
        m = hmm.fit(X, K, seed=K)
        m["states"] = hmm.states(m, X)
        out["models"][K] = m
    return name, out


t0 = time.time()
with ProcessPoolExecutor(4) as ex:
    H = dict(ex.map(fit_recording, sq.MAIN))
iters = pd.DataFrame([dict(rec=k, K=K, iters=m["iters"]) for k, h in H.items() for K, m in h["models"].items()])
print(f"fitted {len(iters)} HMMs in {time.time() - t0:.0f} s")

In [ ]:
print(pd.DataFrame([dict(rec=k, type=h["mtype"], units=h["U"], bins=h["X"].shape[0], bin_ms=hmm.BIN_MS,
                         max_count=int(h["X"].max()), bursts=h["n_bursts"],
                         backbone=int(h["bb"].sum()) if "bb" in h else np.nan) for k, h in H.items()]).to_string(index=False))
print("\nEM iterations per model: median", iters.iters.median(), "; models stopped at the limit of",
      hmm.MAX_ITER, ":", int((iters.iters >= hmm.MAX_ITER).sum()), "of", len(iters))

## supp18-hmm-likelihood-real-above-shuffled

> Hidden Markov models fitted to the organoid spike trains give a higher held-out log likelihood
> on real data than on randomized data with the same unit and population rates.

In [ ]:
HO8 = [f"HO{i}" for i in range(1, 9)]


def cv_recording(args):
    name, frac, ks = args
    M = sq.load_matrix(first.loc[name].asset_id, sq.keep_seconds(name))
    X = hmm.bin_counts(M)
    S = hmm.bin_counts(sq.okun_shuffle(M, 2953)[0])  # the code's randomized surrogate (seed as the code)
    rng = np.random.default_rng(7)
    rows = []
    for K in ks:
        for fold in range(5):
            train = np.ones_like(X, bool)
            flat = rng.choice(X.size, int(X.size * frac), replace=False)
            train.flat[flat] = False
            m = hmm.fit(X, K, seed=fold, mask=train)
            n_held = (~train).sum()
            # both scored the same way, log p(held-out entries | the rest) = log p(all) - log p(rest)
            real = hmm.loglik(m, X) - hmm.loglik(m, X, train)
            sur = hmm.loglik(m, S) - hmm.loglik(m, S, train)
            # the round-1 scoring of the surrogate, the marginal likelihood of its held-out entries alone
            sur_marginal = hmm.loglik(m, S, ~train)
            rows.append(dict(rec=name, K=K, fold=fold, held_out=frac, validation=real / n_held, surrogate=sur / n_held,
                             surrogate_marginal=sur_marginal / n_held))
    return rows


jobs = [(k, 0.2, CV_KS) for k in HO8] + [(k, 0.1, [20]) for k in HO8]
with ProcessPoolExecutor(4) as ex:
    cv = pd.DataFrame([r for rows in ex.map(cv_recording, jobs) for r in rows])

In [ ]:
cv["real_above"] = cv.validation > cv.surrogate
cv["margin"] = cv.validation - cv.surrogate
print(cv.groupby(["held_out", "rec"]).agg(folds=("fold", "size"), real_above=("real_above", "sum"),
                                          validation=("validation", "mean"), surrogate=("surrogate", "mean"),
                                          margin_min=("margin", "min"), margin_max=("margin", "max")).round(4).to_string())
for frac in [0.2, 0.1]:
    c = cv[cv.held_out == frac]
    print(f"held out {frac:.0%}: real above surrogate in {int(c.real_above.sum())} of {len(c)} fold, K and organoid combinations; "
          f"smallest margin {c.margin.min():.4f} nats per entry")
print("\nMargin per fold (nats per entry), 20% held out:")
print(cv[cv.held_out == 0.2].pivot_table(index=["rec", "K"], columns="fold", values="margin").round(4).to_string())
c = cv[cv.held_out == 0.2]
print(f"\nRound-1 scoring (surrogate as a marginal likelihood): smallest margin {(c.validation - c.surrogate_marginal).min():.4f}; "
      f"marginal minus conditional surrogate score, mean {(c.surrogate_marginal - c.surrogate).mean():.4f}")

**Result: reproduced.** Scored the same way on both, as the log likelihood of the held-out
entries given the rest, the real data beat the randomized surrogate in all 120 combinations of
fold, K and organoid with 20% held out (smallest margin 0.015 nats per entry, HO4 at K = 30) and in
all 40 with the code's 10% (smallest 0.021). The round-1 scoring gave the surrogate the marginal
likelihood of its held-out entries, which is lower by 0.008 nats per entry on average; with it the
smallest margin was 0.009.

## fig5c-states-converge-at-peak

> The diversity of HMM states is high before and after bursts and reaches a minimum near the burst
> peak, where states follow a stereotyped path.

In [ ]:
offs = np.arange(-500 // hmm.BIN_MS, 500 // hmm.BIN_MS + 1)
rows = []
for k in HO8:
    h = H[k]
    peaks = h["bursts"]["peak"]
    ent = []
    for K, m in h["models"].items():
        st = m["states"]
        pb = peaks // hmm.BIN_MS
        ok = (pb + offs[0] >= 0) & (pb + offs[-1] < len(st))
        seq = st[pb[ok][:, None] + offs[None, :]]
        e = []
        for j in range(len(offs)):
            p = np.bincount(seq[:, j], minlength=K) / seq.shape[0]
            p = p[p > 0]
            e.append(-(p * np.log2(p)).sum())
        ent.append(e)
    ent = np.mean(ent, 0)
    jmin = int(np.argmin(ent))
    tmin = offs[jmin] * hmm.BIN_MS
    win = h["window"]
    rows.append(dict(rec=k, entropy_start=ent[0], entropy_min=ent[jmin], entropy_end=ent[-1], min_at_ms=tmin,
                     a_within_90ms=abs(tmin) <= 90,
                     b_in_backbone_window=(win is not None and win[0] - hmm.BIN_MS <= tmin <= win[1]),
                     below_both_ends=bool(ent[jmin] < ent[0] and ent[jmin] < ent[-1])))
f5c = pd.DataFrame(rows)
print(f5c.round(3).to_string(index=False))
print("Organoids meeting (a):", int((f5c.a_within_90ms & f5c.below_both_ends).sum()),
      "; meeting (b):", int((f5c.b_in_backbone_window & f5c.below_both_ends).sum()), "of 8")

**Result: not reproduced.** The state entropy across bursts, averaged over K, has its minimum
below both window ends in seven organoids, but within 90 ms of the peak only in HO8, and within
the backbone window in three (HO1, HO6, HO8); in HO2, HO3 and HO4 the minimum lies 120 to 510 ms
before the peak, where the entropy is already near 0 at the window's start in HO3 (0.015) and HO4
(0.18). The criterion asks for at least 6 of 8 under both readings.

## fig5d-backbone-more-states and fig5f-svm-states-beat-rate

In [ ]:
def consistency(m, X):
    """Fraction of each state's realizations (bins where it is the most likely state) in which each
    unit fires at least once (Supplementary Methods): states by units, NaN for unused states."""
    st = m["states"]
    K = m["lam"].shape[0]
    C = np.full((K, X.shape[1]), np.nan)
    for s in range(K):
        sel = st == s
        if sel.any():
            C[s] = (X[sel] > 0).mean(0)
    return C


rows = []
for k in HO8:
    h = H[k]
    if not h["bb"].sum():
        continue
    for K, m in h["models"].items():
        cm = consistency(m, h["X"])
        for u in range(h["U"]):
            rows.append(dict(rec=k, K=K, unit=u, bb=int(h["bb"][u]), n_above=int(np.nansum(cm[:, u] > 0.5)),
                             mean_cons=np.nanmean(cm[:, u])))
cons = pd.DataFrame(rows)

In [ ]:
print(cons.groupby(["rec", "bb"])[["n_above", "mean_cons"]].mean().round(3).unstack().to_string())
for v in ["n_above", "mean_cons"]:
    f = sq.lmm(cons, f"{v} ~ bb + C(K)", "rec")
    print(f"{v}: backbone minus non-rigid {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g} ({len(cons)} unit-model rows, "
          f"{cons.rec.nunique()} organoids; K as a fixed factor)")
print("HO7 has no backbone unit and is left out.")

**Result: reproduced.** Backbone units are consistent (above 0.5) in more states than non-rigid
units (by 3.7 states) and have a higher mean consistency (by 0.17), p printed as 0 for both, in
each of the seven organoids with backbone units.

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler


def loo_accuracy(F, y, seed):
    """Leave-one-out accuracy of a standardised linear SVM (SGDClassifier, hinge loss; Fig5.py)."""
    hits = 0
    for i in range(len(y)):
        tr = np.ones(len(y), bool)
        tr[i] = False
        sc = StandardScaler().fit(F[tr])
        clf = SGDClassifier(random_state=seed).fit(sc.transform(F[tr]), y[tr])
        hits += int(clf.predict(sc.transform(F[i:i + 1]))[0] == y[i])
    return hits / len(y)


rows = []
for k in HO8:
    h = H[k]
    if not h["bb"].sum():
        continue
    y = h["bb"].astype(int)
    rate_acc = loo_accuracy(h["rate"][:, None], y, 0)
    for K, m in h["models"].items():
        feats = np.nan_to_num(consistency(m, h["X"]), nan=0.0).T
        rows.append(dict(rec=k, K=K, state_acc=loo_accuracy(feats, y, 0), rate_acc=rate_acc))
svm = pd.DataFrame(rows)
chance = pd.Series({k: max(H[k]["bb"].mean(), 1 - H[k]["bb"].mean()) for k in svm.rec.unique()}, name="majority_class")
print(svm.groupby("rec")[["state_acc", "rate_acc"]].mean().join(chance).round(3).to_string())
print(f"Chance (majority class) per organoid, mean {100 * chance.mean():.1f}%; the paper states no chance level")
print(f"\nState features: mean {100 * svm.state_acc.mean():.1f}% +- {100 * svm.state_acc.std():.1f} over all models; "
      f"rate: {100 * svm.groupby('rec').rate_acc.first().mean():.1f}% +- {100 * svm.groupby('rec').rate_acc.first().std():.1f} over organoids")
per = svm.groupby("rec").agg(state=("state_acc", "mean"), rate=("rate_acc", "first"))
print("(a) paired t-test over organoids:", stats.ttest_rel(per.state, per.rate))
pb = {k: stats.ttest_1samp(g.state_acc, g.rate_acc.iloc[0]) for k, g in svm.groupby("rec")}
print("(b) per organoid one-sample t-test of state accuracies against the rate accuracy:")
for k, t in pb.items():
    print(f"   {k}: t {t.statistic:.1f}, p {t.pvalue:.2g}")
print("Expected: states 83.9 +- 12.0%, rate 63.2 +- 10.4%")

**Result: not reproduced.** The state-feature accuracy, 86.5% ± 11.6% over all models, is
within 83.9 ± 12.0%, but the rate-only accuracy, 73.9% ± 23.2% over seven organoids, is outside
63.2 ± 10.4%, and states beat rate significantly in neither test: paired over organoids p =
0.086; per organoid in four of seven (HO2, HO3 and HO4, where rate alone reaches 89% to 95%, are
not). The majority class is 54% to 88% of units (mean 70.6%); the rate-only accuracy is below it in
HO5 and HO8. HO7 has no backbone unit to classify.
The criterion defines no partial outcome, and its central comparison, states against rate, fails
under both readings, so the claim is not reproduced (the rule for partial outcomes is in the
summary of the main notebook).

## fig7f-hmm-dimensionality

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

rows = []
for k, h in H.items():
    for K, m in h["models"].items():
        L = m["lam"] / (hmm.BIN_MS / 1000.0)
        Lc = L - L.mean(0)
        s = np.linalg.svd(Lc, compute_uv=False) ** 2
        cum = np.cumsum(s) / s.sum()
        rows.append(dict(rec=k, mtype=h["mtype"], K=K, dim=int(min(np.searchsorted(cum, 0.75) + 1, 10))))
dim = pd.DataFrame(rows)
print(dim.groupby(["mtype", "rec"]).dim.mean().round(2).unstack(0).to_string())
print(dim.groupby("mtype").dim.mean().round(2).to_dict())
gee = smf.gee("dim ~ 0 + C(mtype)", "rec", dim, family=sm.families.Poisson(), cov_struct=sm.cov_struct.Exchangeable()).fit()
names = list(gee.params.index)
cov = gee.cov_params()
for a, b in [("PR", "HO"), ("PR", "MO"), ("HO", "MS"), ("MO", "MS")]:
    w = np.array([1.0 if n == f"C(mtype)[{a}]" else -1.0 if n == f"C(mtype)[{b}]" else 0.0 for n in names])
    est = float(w @ gee.params.values)
    se = float(np.sqrt(w @ cov.values @ w))
    print(f"{a} minus {b} (log scale): {est:.3f}, p {2 * stats.norm.sf(abs(est / se)):.2g}")
print("Test: Poisson GEE with an exchangeable correlation within recording (the paper: glmer with a random intercept).")

**Result: reproduced.** HMM dimensionality at theta = 0.75 is lower in primary cultures (mean
1.73) than in human (2.85, p = 3.5e-6) and murine organoids (2.59, p = 0.005). Human organoids and
slices do not differ here (p = 0.60), where the paper reports P = 0.04 (not judged).

## supp24d-backbone-non-poisson

In [ ]:
rows = []
for k, h in H.items():
    if "bb" not in h or "bursts" not in h:
        continue
    X = h["X"]
    inb = np.zeros(X.shape[0], bool)
    for s0, e0 in zip(h["bursts"]["start"], h["bursts"]["end"]):
        inb[s0 // hmm.BIN_MS:e0 // hmm.BIN_MS + 1] = True
    frac = np.zeros(h["U"])
    wsum = 0.0
    for K, m in h["models"].items():
        st = m["states"]
        for s in range(K):
            sel = (st == s) & inb
            n = int(sel.sum())
            if n < 2:
                continue
            x = X[sel]
            mu = x.mean(0)
            var = x.var(0, ddof=1)
            with np.errstate(invalid="ignore", divide="ignore"):
                stat = (n - 1) * var / mu
            p = stats.chi2.cdf(stat, n - 1)
            nonp = np.where(mu > 0, p < 0.01, False)
            frac += n * nonp
            wsum += n
    frac /= max(wsum, 1)
    rows += [dict(rec=k, mtype=h["mtype"], bb=int(h["bb"][u]), frac=frac[u]) for u in range(h["U"])]
npu = pd.DataFrame(rows)
med = npu.groupby(["mtype", "bb"]).frac.median().unstack()
print(med.round(3).to_string())
print("Backbone median above non-rigid in every type:", bool((med[1] > med[0]).all()),
      "; non-rigid median at most 0.1 in every type:", bool((med[0] <= 0.1).all()))
f = sq.lmm(npu, "frac ~ bb", "rec")
print(f"Mixed model, backbone minus non-rigid: {f.params['bb']:.3f}, p {f.pvalues['bb']:.2g}")

**Result: reproduced.** In every model type the median fraction of non-Poisson states is 0 or
0.001 for non-rigid units and higher for backbone units (0.013 to 0.262); backbone minus
non-rigid 0.21 in the mixed model, p printed as 0.

## supp26-state-traversal

In [ ]:
rows = []
for k, h in H.items():
    if h.get("window") is None:
        continue
    start, stop = h["window"]
    length = stop - start
    if length <= 0:
        continue
    nb_ = int(np.ceil(length / hmm.BIN_MS))
    for K, m in h["models"].items():
        st = m["states"]
        counts = []
        for pk in h["bursts"]["peak"]:
            b0 = int((pk + start) / hmm.BIN_MS)
            seq = st[b0:b0 + nb_]
            if len(seq):
                counts.append(len(set(seq)))
        rows.append(dict(rec=k, mtype=h["mtype"], K=K, rate=np.mean(counts) / length * 1e3))
tr = pd.DataFrame(rows)
print(tr.groupby("mtype").rate.describe().round(2).to_string())
f = sq.lmm(tr, "rate ~ 0 + C(mtype) + C(K)", "rec")
for a, b in [("PR", "HO"), ("PR", "MS"), ("HO", "MO"), ("PR", "MO")]:
    c = sq.contrast(f, {f"C(mtype)[{a}]": 1.0, f"C(mtype)[{b}]": -1.0})
    print(f"{a} minus {b}: {c['estimate']:.2f} states per s, p {c['p']:.2g}")
print("Recordings:", tr.groupby("mtype").rec.nunique().to_dict())

**Result: partially reproduced.** Primary cultures traverse more states per second in the
backbone window than human organoids (by 19.8, p = 9e-5), and human and murine organoids do not
differ (4.5, p = 0.35), but primary cultures are above slices by 8.8 with p = 0.056, not below
0.05. Primary cultures also exceed murine organoids (p = 2.5e-5), where the paper reports P =
0.064. The p-values are not Tukey-adjusted; an adjustment would only raise them.
Two of the three judged contrasts hold and the third has the paper's direction but falls short of
0.05, so the claim holds in part (the same rule as in the main notebook's summary).

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| supp18-hmm-likelihood-real-above-shuffled | reproduced |
| fig5c-states-converge-at-peak | not reproduced |
| fig5d-backbone-more-states | reproduced |
| fig5f-svm-states-beat-rate | not reproduced |
| fig7f-hmm-dimensionality | reproduced |
| supp24d-backbone-non-poisson | reproduced |
| supp26-state-traversal | partially reproduced |

**Differences from the paper's method.** HMMs fitted with this entry's EM in place of SSM, with
K = 10, 12, ..., 30 in place of every K from 10 to 30 and at most 300 iterations in place of 5000
(41 of 341 models stopped there); cross-validation folds refitted from scratch with K = 10, 20 and
30; the Poisson GLMM of Fig. 7F replaced by a Poisson GEE with exchangeable correlation within
recording; for fig5d, K enters as a fixed factor in place of a random intercept; mixed models
tested with Satterthwaite t (`sequences.lmm`) without the Tukey adjustment of emmeans.